# 视觉算法面试手撕代码题库

涵盖 Python 基础、NumPy、PyTorch、CNN 组件、目标检测、图像处理等高频考点。

---
## 目录
1. [Python 基础](#1-python-基础)
2. [NumPy 操作](#2-numpy-操作)
3. [PyTorch 基础](#3-pytorch-基础)
4. [CNN 基础组件](#4-cnn-基础组件)
5. [经典网络模块](#5-经典网络模块)
6. [损失函数手写](#6-损失函数手写)
7. [目标检测组件](#7-目标检测组件)
8. [注意力机制](#8-注意力机制)
9. [数据增强与预处理](#9-数据增强与预处理)
10. [评价指标](#10-评价指标)
11. [训练流程](#11-训练流程)
12. [综合实战](#12-综合实战)

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
from collections import OrderedDict

---
## 1. Python 基础

### 1.1 手写快速排序

**核心原理：** 选择一个基准元素(pivot)，将数组分为"小于pivot"、"等于pivot"、"大于pivot"三部分，递归排序。

**时间复杂度：**
- 平均 O(n log n)，最坏 O(n²)（数组已排序且pivot选最端点时）
- 空间复杂度：非原地版 O(n)，原地版 O(log n)（递归栈）

**面试要点：**
1. pivot 选择策略：选中间元素比选首/尾更优，可避免已排序数组的最坏情况
2. 原地版使用 Lomuto 分区方案（代码中的 `partition` 函数），维护一个分界指针 `i`
3. 原地版是面试加分项，展示对算法的深入理解

**注意事项：**
- 非原地版本虽然简洁易懂，但每次递归都创建新列表，空间开销大
- `partition` 中 `arr[high]` 作为 pivot，遍历时将 ≤ pivot 的元素交换到左侧
- 最后将 pivot 放到正确位置 `i+1`，返回该位置作为分区点

In [ ]:
def quick_sort(arr):
    """手写快速排序"""
    if len(arr) <= 1:
        return arr
    pivot = arr[len(arr) // 2]
    left = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    return quick_sort(left) + middle + quick_sort(right)

print(quick_sort([3, 6, 8, 10, 1, 2, 1]))

# 原地版本（面试加分）
def quick_sort_inplace(arr, low=0, high=None):
    if high is None:
        high = len(arr) - 1
    if low < high:
        pi = partition(arr, low, high)
        quick_sort_inplace(arr, low, pi - 1)
        quick_sort_inplace(arr, pi + 1, high)

def partition(arr, low, high):
    pivot = arr[high]
    i = low - 1
    for j in range(low, high):
        if arr[j] <= pivot:
            i += 1
            arr[i], arr[j] = arr[j], arr[i]
    arr[i + 1], arr[high] = arr[high], arr[i + 1]
    return i + 1

arr = [3, 6, 8, 10, 1, 2, 1]
quick_sort_inplace(arr)
print("原地:", arr)

### 1.2 手写 Top-K（堆排序思路）

**核心原理：** 维护一个大小为 K 的**小顶堆**，遍历数组时：
- 堆未满 → 直接插入
- 堆已满且当前元素 > 堆顶 → 替换堆顶（堆顶是当前 K 个最大元素中的最小值）

**时间复杂度：** O(n log k)，远优于排序后取前 K 的 O(n log n)

**面试要点：**
1. 为什么用**小顶堆**找最大 K 个？因为堆顶是最小值，新元素只需和最小值比较
2. `heapq` 是 Python 内置的小顶堆，`heapreplace` 先弹出再插入，比 `heappop + heappush` 更高效
3. 面试常问：海量数据（内存放不下）时如何找 Top-K？→ 正是用堆的方法，逐条读入

**注意事项：**
- `heapq.nlargest(k, arr)` 是库函数直接调用，面试时需要手写堆逻辑
- 如果要找最小的 K 个元素，应使用**大顶堆**（或对元素取负数用小顶堆模拟）
- `heapreplace` 等价于先 `heappop` 再 `heappush`，但效率更高（少一次调整）

In [ ]:
import heapq

def top_k(arr, k):
    """返回数组中最大的k个元素"""
    return heapq.nlargest(k, arr)

# 手写小顶堆版本
def top_k_manual(arr, k):
    heap = []
    for num in arr:
        if len(heap) < k:
            heapq.heappush(heap, num)
        elif num > heap[0]:
            heapq.heapreplace(heap, num)
    return sorted(heap, reverse=True)

arr = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5]
print(top_k(arr, 3))
print(top_k_manual(arr, 3))

### 1.3 二分查找（及其变体）

**核心原理：** 在**有序数组**中，每次将搜索范围缩小一半。

**时间复杂度：** O(log n)

**面试要点：**
1. **循环条件**：标准版用 `left <= right`（找精确匹配），`lower_bound` 用 `left < right`（找边界）
2. **中点计算**：`mid = (left + right) // 2`，Python 不会溢出，但 C++/Java 中应写 `left + (right - left) // 2`
3. **变体很重要**：
   - `lower_bound`：第一个 ≥ target 的位置（C++ STL 常考）
   - `upper_bound`：第一个 > target 的位置
   - 这两个变体是解决"在排序数组中查找元素的首尾位置"的基础

**注意事项：**
- 标准版 `right = len(arr) - 1`（闭区间），`lower_bound` 用 `right = len(arr)`（左闭右开）
- `lower_bound` 中 `right` 的更新是 `right = mid`（不是 `mid - 1`），因为 `mid` 本身可能是答案
- 二分查找前提是数组**已排序**，面试时先确认这个条件

In [ ]:
def binary_search(arr, target):
    """标准二分查找，返回索引，未找到返回 -1"""
    left, right = 0, len(arr) - 1
    while left <= right:
        mid = (left + right) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1

def lower_bound(arr, target):
    """第一个 >= target 的位置"""
    left, right = 0, len(arr)
    while left < right:
        mid = (left + right) // 2
        if arr[mid] < target:
            left = mid + 1
        else:
            right = mid
    return left

arr = [1, 2, 3, 4, 5, 6, 7, 8, 9]
print(binary_search(arr, 5))   # 4
print(lower_bound(arr, 5))     # 4
print(lower_bound(arr, 5.5))   # 5

### 1.4 生成器与迭代器

**核心原理：** `yield` 使函数变为生成器，**惰性求值**——每次调用 `next()` 才计算下一个值，不一次性生成所有结果。

**面试要点：**
1. **斐波那契生成器**：经典面试题，用 `yield` 实现无限/有限序列
   - `a, b = b, a + b` 是 Python 的元组解包赋值，同时更新，无需临时变量
2. **批量迭代器**：模拟 PyTorch `DataLoader` 的核心思路
   - 支持 shuffle：先打乱索引，再按 batch_size 分批
   - `yield` 使得每次只返回一个 batch，内存友好
3. 生成器 vs 列表：生成器**省内存**（不存储全部结果），但只能遍历一次

**注意事项：**
- 生成器是**一次性**的，遍历完后再次遍历会得到空结果
- 面试中常问：`yield` 和 `return` 的区别？→ `yield` 暂停函数状态，下次调用从暂停处继续；`return` 终止函数
- `batch_iterator` 中最后一个 batch 可能不满 `batch_size`，DataLoader 中可用 `drop_last=True` 丢弃

In [ ]:
# 斐波那契生成器 — 面试常考
def fibonacci(n):
    a, b = 0, 1
    for _ in range(n):
        yield a
        a, b = b, a + b

print(list(fibonacci(10)))

# 批量数据迭代器（模拟 DataLoader 的核心思路）
def batch_iterator(data, batch_size, shuffle=True):
    indices = list(range(len(data)))
    if shuffle:
        random.shuffle(indices)
    for i in range(0, len(indices), batch_size):
        batch_indices = indices[i:i + batch_size]
        yield [data[j] for j in batch_indices]

for batch in batch_iterator(list(range(20)), batch_size=6):
    print(batch)

### 1.5 LRU Cache 手写

**核心原理：** LRU (Least Recently Used) 最近最少使用缓存淘汰策略。容量满时淘汰最久未被访问的数据。

**数据结构选择：**
- `OrderedDict`：Python 内置的有序字典，`move_to_end(key)` 将键移到末尾（标记为最近使用），`popitem(last=False)` 弹出头部（最久未使用）
- 面试高级版：手写双向链表 + 哈希表，O(1) 的 get/put

**时间复杂度：** get 和 put 均 O(1)

**面试要点：**
1. 操作流程：get 时先移到末尾再返回；put 时若已存在则先移到末尾再更新，超容量则弹出头部
2. `OrderedDict` 底层就是双向链表 + 哈希表的实现
3. Python 标准库 `functools.lru_cache` 是装饰器版本，基于同样的原理

**注意事项：**
- `popitem(last=False)` 弹出的是**最早插入且未被访问**的项（FIFO 头部）
- `move_to_end(key)` 是维护"最近使用"顺序的关键操作，get 和 put 都要调用
- 面试时可能要求用链表手写，需要维护 `prev`/`next` 指针和 `head`/`tail` 虚拟节点

In [ ]:
from collections import OrderedDict

class LRUCache:
    def __init__(self, capacity):
        self.cache = OrderedDict()
        self.capacity = capacity

    def get(self, key):
        if key not in self.cache:
            return -1
        self.cache.move_to_end(key)  # 移到末尾表示最近使用
        return self.cache[key]

    def put(self, key, value):
        if key in self.cache:
            self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) > self.capacity:
            self.cache.popitem(last=False)  # 弹出最久未使用的

lru = LRUCache(3)
lru.put(1, 'a')
lru.put(2, 'b')
lru.put(3, 'c')
print(lru.get(1))  # a
lru.put(4, 'd')    # 淘汰 key=2
print(lru.get(2))  # -1 (已淘汰)

---
## 2. NumPy 操作

### 2.1 Softmax 手写（数值稳定版）

**核心公式：** $\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$

**数值稳定性关键：** 先减去最大值 `x - max(x)`，再求 exp。这不会改变 softmax 的结果（因为分子分母同时除以 $e^{\max}$），但能防止 `exp(大数)` 导致的数值溢出。

**面试要点：**
1. **为什么需要数值稳定？** 当 x 中有 1000 这样的值时，`exp(1000)` 会超出 float64 范围变成 `inf`，导致 `nan`
2. 减去 `max(x)` 后，最大的 exp 值为 `exp(0) = 1`，其余均 < 1，不会溢出
3. `keepdims=True` 保持维度，确保广播正确（如对 (N, C) 的每一行做 softmax）
4. `axis=-1` 表示沿最后一个维度操作，适用于任意形状的输入

**注意事项：**
- Softmax 输出之和严格等于 1（概率分布）
- 数值稳定版与原始版本**数学上等价**，只是避免了浮点溢出
- 配合 log 使用时，应直接用 `log_softmax`（比 `log(softmax(x))` 更稳定）

In [ ]:
def softmax(x):
    """数值稳定的 softmax"""
    x = x - np.max(x, axis=-1, keepdims=True)  # 防止溢出
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

x = np.array([1000, 1001, 1002])
print(softmax(x))
print("sum:", softmax(x).sum())

### 2.2 卷积手写（2D 卷积，无 padding）

**核心操作：** 卷积核（kernel/filter）在输入上滑动，每个位置做**逐元素相乘再求和**，得到输出的一个像素。

**输出尺寸公式（valid padding）：** $O = I - K + 1$

**面试要点：**
1. **卷积 vs 互相关**：严格来说这里实现的是互相关（cross-correlation），数学上的卷积需要先翻转核。但深度学习中"卷积"实际指的就是互相关
2. 输出每个元素的计算：`output[i,j] = Σ(input[i:i+kh, j:j+kw] * kernel)`
3. 这是 **valid padding**（无填充），输出比输入小。若要保持尺寸需用 same padding（补零）

**注意事项：**
- 此处为**单通道、单核**的简化版本。实际 CNN 中：
  - 多输入通道：核的通道数 = 输入通道数，各通道结果相加
  - 多输出通道：使用多个核，每个核产生一个输出通道
- 时间复杂度：O(H × W × KH × KW)，实际中使用 im2col + 矩阵乘法加速
- 代码中的 `*` 是 NumPy 的逐元素乘法（Hadamard 积），不是矩阵乘法

In [ ]:
def conv2d(input_mat, kernel):
    """手写 2D 卷积 (valid padding)"""
    h, w = input_mat.shape
    kh, kw = kernel.shape
    oh, ow = h - kh + 1, w - kw + 1
    output = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            output[i, j] = np.sum(input_mat[i:i+kh, j:j+kw] * kernel)
    return output

input_mat = np.array([[1, 2, 3, 0],
                       [0, 1, 2, 3],
                       [3, 0, 1, 2],
                       [2, 3, 0, 1]], dtype=np.float32)

kernel = np.array([[0, 1, 0],
                    [1, -4, 1],
                    [0, 1, 0]], dtype=np.float32)  # Laplacian

print(conv2d(input_mat, kernel))

### 2.3 Max Pooling 手写

**核心操作：** 在每个池化窗口内取最大值，实现**下采样**，减小特征图尺寸的同时保留最显著特征。

**输出尺寸公式：** $O = \lfloor \frac{I - K}{S} \rfloor + 1$

**面试要点：**
1. Max Pooling 的作用：降低空间分辨率、增加感受野、提供一定程度的平移不变性
2. 常用配置：`kernel_size=2, stride=2`（尺寸减半）
3. 与 Avg Pooling 的区别：Max Pooling 保留最显著特征，Avg Pooling 保留平均信息
4. 现代网络趋势：用 `stride>1` 的卷积替代 pooling（如 ResNet），或用 `AdaptiveAvgPool` 替代固定大小的 pooling

**注意事项：**
- 当 `stride < kernel_size` 时，池化窗口会**重叠**
- 当输入尺寸不能被 stride 整除时，边缘可能被丢弃（Ceil 模式 vs Floor 模式）
- Max Pooling **没有可学习参数**，仅做固定操作
- 反向传播时，梯度只流向最大值所在位置（其余位置梯度为 0）

In [ ]:
def max_pool2d(input_mat, pool_size=2, stride=2):
    h, w = input_mat.shape
    oh = (h - pool_size) // stride + 1
    ow = (w - pool_size) // stride + 1
    output = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            si, sj = i * stride, j * stride
            output[i, j] = np.max(input_mat[si:si+pool_size, sj:sj+pool_size])
    return output

mat = np.array([[1, 2, 3, 4],
                [5, 6, 7, 8],
                [9, 10, 11, 12],
                [13, 14, 15, 16]], dtype=np.float32)

print(max_pool2d(mat))

### 2.4 Batch Normalization 手写

**核心公式：**
$$y = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

- $\mu, \sigma^2$：沿 batch 维度计算的均值和方差
- $\gamma, \beta$：可学习的缩放和偏移参数（恢复表达能力）
- $\epsilon$：小常数（如 1e-5），防止除零

**面试要点：**
1. **BN 的作用**：缓解 Internal Covariate Shift，允许使用更大学习率，加速收敛，有轻微正则化效果
2. **训练 vs 推理**：
   - 训练时：使用当前 batch 的均值/方差（代码中的实现）
   - 推理时：使用训练过程中累计的**全局均值/方差**（running mean/var）
3. **BN 的位置**：通常在 Conv → BN → ReLU 的顺序中使用
4. 对于 (N, C, H, W) 输入，BN 沿 (N, H, W) 三个维度计算统计量，每个通道独立

**注意事项：**
- **小 batch size** 时 BN 效果差（统计量不稳定），可考虑 Group Norm 或 Layer Norm
- $\gamma$ 初始化为 1，$\beta$ 初始化为 0
- BN 依赖 batch 统计量，因此**训练和推理行为不同**，这是面试常见追问点
- 这里是简化实现，完整版还需要 running mean/var 的指数移动平均更新

In [ ]:
def batch_norm(x, gamma=1.0, beta=0.0, eps=1e-5):
    """
    x: shape (N, C, H, W) 或 (N, D)
    沿 batch 维度计算均值和方差
    """
    mean = np.mean(x, axis=0)
    var = np.var(x, axis=0)
    x_norm = (x - mean) / np.sqrt(var + eps)
    return gamma * x_norm + beta

# 测试
x = np.random.randn(4, 3, 32, 32)
out = batch_norm(x, gamma=1.0, beta=0.0)
print("mean per channel:", out.mean(axis=(0, 2, 3)).round(4))
print("var  per channel:", out.var(axis=(0, 2, 3)).round(4))

---
## 3. PyTorch 基础

### 3.1 Tensor 操作速查

**核心概念：** Tensor 是 PyTorch 的核心数据结构，类似 NumPy ndarray，但支持 GPU 加速和自动求导。

**面试高频操作分类：**

| 类别 | 操作 | 说明 |
|------|------|------|
| 创建 | `zeros`, `ones`, `randn`, `arange` | 初始化 tensor |
| 形状 | `view/reshape`, `permute`, `transpose` | 改变形状/维度顺序 |
| 广播 | 不同形状 tensor 运算 | 自动扩展维度 |
| 拼接 | `cat`, `stack` | 合并 tensor |

**关键区别（面试常问）：**
1. **`view` vs `reshape`**：`view` 要求内存连续（contiguous），`reshape` 不要求（必要时会复制）
2. **`cat` vs `stack`**：`cat` 沿已有维度拼接（不增加维度），`stack` 沿新维度堆叠（增加一个维度）
3. **`permute` vs `transpose`**：`permute` 可重排任意多维，`transpose` 只交换两个维度

**注意事项：**
- NCHW（batch, channel, height, width）是 PyTorch 的默认图像格式
- 广播规则：从最右维度开始比较，大小为 1 或相同的维度可广播
- `view(-1)` 中的 `-1` 表示自动推断该维度大小

In [ ]:
# 创建
a = torch.zeros(2, 3)
b = torch.ones(2, 3)
c = torch.randn(2, 3)
d = torch.arange(12).reshape(3, 4)

# 形状操作
x = torch.randn(4, 3, 32, 32)
print(x.shape)
print(x.view(4, 3, -1).shape)          # 展平空间维度
print(x.permute(0, 2, 3, 1).shape)     # NCHW -> NHWC
print(x.transpose(1, 2).shape)         # 交换 dim1, dim2

# 广播
a = torch.randn(4, 1, 3)
b = torch.randn(1, 5, 3)
print((a + b).shape)  # (4, 5, 3)

# 拼接
a = torch.randn(2, 3)
b = torch.randn(2, 3)
print(torch.cat([a, b], dim=0).shape)  # (4, 3)
print(torch.cat([a, b], dim=1).shape)  # (2, 6)
print(torch.stack([a, b], dim=0).shape)  # (2, 2, 3)

### 3.2 autograd 基本流程

**核心原理：** PyTorch 的自动微分引擎。对 `requires_grad=True` 的 tensor 执行运算，PyTorch 会记录计算图，调用 `.backward()` 时自动计算梯度。

**计算步骤：**
1. 定义参数 `w, b`，设置 `requires_grad=True`
2. 前向传播：计算预测值和损失
3. `loss.backward()`：反向传播，自动计算 dL/dw 和 dL/db
4. 在 `torch.no_grad()` 下更新参数（不记录这次操作到计算图）
5. 清零梯度 `grad.zero_()`（PyTorch 默认梯度是累加的）

**面试要点：**
1. **为什么梯度要清零？** PyTorch 默认 `grad += new_grad`（累加），不清零会导致梯度不断叠加
2. **`torch.no_grad()` 的作用**：参数更新操作不应被记录到计算图中，否则会内存泄漏
3. **计算图的生命周期**：`backward()` 后计算图默认被释放，再次 `backward()` 会报错（除非设 `retain_graph=True`）

**注意事项：**
- 只有设置了 `requires_grad=True` 的 tensor 才会追踪梯度
- `.item()` 将单元素 tensor 转为 Python 标量，用于打印或记录
- 这个例子用 SGD 手动更新参数，实际中用 `optimizer.step()` 封装

In [ ]:
# 简单的线性回归 autograd 示例
torch.manual_seed(42)

# y = 3x + 2
x = torch.randn(100, 1)
y = 3 * x + 2 + 0.1 * torch.randn(100, 1)

w = torch.randn(1, 1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

lr = 0.1
for step in range(100):
    pred = x @ w + b
    loss = ((pred - y) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad.zero_()
    b.grad.zero_()

print(f"w={w.item():.3f}, b={b.item():.3f}")  # 应接近 3 和 2

### 3.3 自定义 Dataset & DataLoader

**核心机制：**
- `Dataset`：定义数据的访问方式（`__len__` + `__getitem__`）
- `DataLoader`：负责 batching、shuffling、多进程加载（`num_workers`）

**面试要点：**
1. **必须实现的两个方法**：
   - `__len__`：返回数据集大小，`len(dataset)` 时调用
   - `__getitem__`：通过索引返回单个样本，`dataset[idx]` 时调用
2. **DataLoader 的关键参数**：
   - `batch_size`：每批样本数
   - `shuffle=True`：每轮 epoch 打乱顺序（训练集用 True，验证/测试集用 False）
   - `num_workers`：多进程加载的进程数（Windows 上建议设为 0 避免多进程问题）
   - `drop_last=True`：丢弃最后一个不完整的 batch
3. **transform 的设计模式**：通过可调用对象（函数/类）对数据进行预处理，灵活组合

**注意事项：**
- `__getitem__` 返回的是单个样本，DataLoader 自动将多个样本堆叠成 batch
- 实际项目中常用 `torchvision.transforms` 做 transform pipeline
- `collate_fn` 可自定义如何将多个样本合并为 batch（如变长序列处理）

In [ ]:
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

# 模拟数据
images = torch.randn(100, 3, 32, 32)
labels = torch.randint(0, 10, (100,))

dataset = MyDataset(images, labels)
loader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=0, drop_last=True)

for batch_imgs, batch_labels in loader:
    print(batch_imgs.shape, batch_labels.shape)
    break

---
## 4. CNN 基础组件

### 4.1 卷积输出尺寸计算

**通用公式（含 dilation）：**
$$O = \lfloor \frac{I + 2P - D \times (K - 1) - 1}{S} \rfloor + 1$$

- $I$：输入尺寸，$K$：卷积核大小，$P$：padding，$S$：stride，$D$：dilation

**常见配置速查：**
| 配置 | 条件 | 输出 |
|------|------|------|
| Same padding | K=3, P=1, S=1 | 尺寸不变 |
| 减半 | K=3, P=1, S=2 | 尺寸减半 |
| Valid | P=0, S=1 | O = I - K + 1 |
| ResNet conv1 | I=224, K=7, P=3, S=2 | 112×112 |

**面试要点：**
1. **dilation（空洞卷积）**：在核元素之间插入空洞，等效感受野增大但不增加参数量。有效核大小 = D × (K - 1) + 1
2. **Same padding 的计算**：P = (K - 1) / 2（当 K 为奇数时刚好对称）
3. 此函数支持 `int` 或 `tuple` 输入（方形或矩形）

**注意事项：**
- 面试中常让手算：如输入 224×224，经过 3 个 stride=2 的层后尺寸是多少？→ 224 → 112 → 56 → 28
- `dilation` 默认为 1（即普通卷积），在 DeepLab 等语义分割网络中常用 dilation > 1

In [ ]:
def conv_output_size(input_size, kernel_size, stride=1, padding=0, dilation=1):
    """计算卷积输出尺寸"""
    if isinstance(kernel_size, int):
        kernel_size = (kernel_size, kernel_size)
    if isinstance(input_size, int):
        input_size = (input_size, input_size)
    if isinstance(stride, int):
        stride = (stride, stride)
    if isinstance(padding, int):
        padding = (padding, padding)
    if isinstance(dilation, int):
        dilation = (dilation, dilation)
    
    h = (input_size[0] + 2 * padding[0] - dilation[0] * (kernel_size[0] - 1) - 1) // stride[0] + 1
    w = (input_size[1] + 2 * padding[1] - dilation[1] * (kernel_size[1] - 1) - 1) // stride[1] + 1
    return h, w

print(conv_output_size(224, 7, stride=2, padding=3))  # ResNet: (112, 112)
print(conv_output_size(112, 3, stride=1, padding=1))  # (112, 112) same padding
print(conv_output_size(112, 3, stride=2, padding=1))  # (56, 56)

### 4.2 感受野计算

**核心概念：** 感受野 (Receptive Field) 是输出特征图上一个像素对应的输入图像上的区域大小。

**递推公式：**
$$RF_i = RF_{i-1} + (K_i - 1) \times Jump_{i-1}$$
$$Jump_i = Jump_{i-1} \times S_i$$

- $RF$：感受野大小，$Jump$：一个输出像素对应输入的步长（累积 stride）
- 初始：$RF_0 = 1$, $Jump_0 = 1$

**面试要点：**
1. **感受野的意义**：RF 越大，该层能"看到"的输入范围越大，越能捕捉全局信息
2. **增加感受野的方式**：堆叠 3×3 卷积（如 VGG）、使用大核、增大 stride、使用 dilation
3. 两个 3×3 卷积的感受野 = 一个 5×5，但参数更少（2×3²=18 < 5²=25）
4. **有效感受野**：理论 RF 内各像素的贡献呈高斯分布，中心区域贡献最大

**注意事项：**
- RF 只考虑了卷积和 pooling，激活函数不影响感受野
- 网络越深，RF 越大，但不等于"越大越好"——需匹配任务需要的上下文范围
- 面试中可能问"ResNet-18 最后一层的感受野是多少"，需要按层逐步计算

In [ ]:
def receptive_field(layers):
    """
    layers: list of (kernel_size, stride, padding)
    返回每层的感受野大小和步长(jump)
    """
    rf = 1
    jump = 1  # 当前层一个像素对应输入的步长
    print(f"{'Layer':<6} {'Kernel':<8} {'Stride':<8} {'RF':<8} {'Jump':<8}")
    for i, (k, s, p) in enumerate(layers):
        rf = rf + (k - 1) * jump
        jump = jump * s
        print(f"{i:<6} {k:<8} {s:<8} {rf:<8} {jump:<8}")
    return rf

# ResNet-18 前4个 stage
layers = [
    (7, 2, 3),   # conv1
    (3, 2, 0),   # maxpool
    (3, 1, 1),   # layer1 conv
    (3, 2, 1),   # layer2 conv (stride=2)
]
rf = receptive_field(layers)

---
## 5. 经典网络模块

### 5.1 ResNet Bottleneck Block

**核心结构：** 1×1 → 3×3 → 1×1 的三层卷积 + 残差连接（shortcut/skip connection）

**设计动机：**
- **1×1 降维**：将高维通道压缩到 `mid_channels`，减少 3×3 卷积的计算量
- **3×3 卷积**：在低维空间做空间特征提取
- **1×1 升维**：恢复到 `mid_channels × expansion`（通常 expansion=4）的高维通道
- **残差连接**：`out += identity`，解决深层网络的梯度消失问题

**面试要点：**
1. **Bottleneck vs BasicBlock**：ResNet-18/34 用 BasicBlock（两个 3×3），ResNet-50/101/152 用 Bottleneck
2. **downsample 分支**：当 stride>1 或通道数变化时，需要对 shortcut 做 1×1 卷积 + BN 对齐维度
3. **残差连接的数学本质**：学习残差 F(x) = H(x) - x，而非直接学习 H(x)。当 F(x)=0 时网络等价于恒等映射，不会退化
4. **参数量**：Bottleneck 的 3×3 卷积在低维操作，比直接在高维做 3×3 节省大量计算

**注意事项：**
- 最后一层（conv3 → bn3）之后**没有 ReLU**，ReLU 在加法之后
- `bias=False` 因为后面紧跟 BN，BN 有偏移参数 β，卷积的 bias 是冗余的

In [ ]:
class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_channels, mid_channels, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, mid_channels, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(mid_channels)
        self.conv2 = nn.Conv2d(mid_channels, mid_channels, 3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(mid_channels)
        self.conv3 = nn.Conv2d(mid_channels, mid_channels * self.expansion, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(mid_channels * self.expansion)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        return F.relu(out)

# 测试
block = Bottleneck(64, 64)
x = torch.randn(1, 64, 56, 56)
print(block(x).shape)  # (1, 256, 56, 56)

### 5.2 MobileNet V2 倒残差模块 (Inverted Residual)

**核心结构：** 与 ResNet Bottleneck **相反**的维度变化：低维 → 高维 → 低维

**三个阶段：**
1. **1×1 升维**（Expand）：低维输入 → `expand_ratio` 倍的高维空间（默认 6 倍）
2. **3×3 深度可分离卷积**（Depthwise）：`groups=hidden_dim` 即每个通道独立卷积，轻量化
3. **1×1 降维**（Project）：高维 → 低维输出，**注意这里没有 ReLU**（线性瓶颈）

**面试要点：**
1. **倒残差 vs 标准残差**：ResNet 先降维再升维，MobileNetV2 先升维再降维——因为深度卷积在低维空间表现差（信息太少）
2. **为什么最后一个 1×1 没有 ReLU？** ReLU 会丢失低维空间中的信息（线性瓶颈 Linear Bottleneck 原理），线性变换能更好地保留信息
3. **ReLU6**：`min(max(x, 0), 6)`，专为移动端设计，在 float16 精度下更鲁棒
4. **残差连接条件**：仅当 stride=1 且输入输出通道数相同时使用

**注意事项：**
- 深度可分离卷积 `groups=channels` 的参数量仅为标准卷积的 1/C_out
- MobileNet 系列的核心思想：用精度换取速度和轻量化

In [ ]:
class InvertedResidual(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, expand_ratio=6):
        super().__init__()
        hidden_dim = in_channels * expand_ratio
        self.use_residual = stride == 1 and in_channels == out_channels
        
        layers = []
        if expand_ratio != 1:
            # 1x1 升维
            layers += [
                nn.Conv2d(in_channels, hidden_dim, 1, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.ReLU6(inplace=True),
            ]
        # 3x3 深度可分离卷积
        layers += [
            nn.Conv2d(hidden_dim, hidden_dim, 3, stride=stride, padding=1,
                      groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU6(inplace=True),
            # 1x1 降维 (无 ReLU)
            nn.Conv2d(hidden_dim, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        ]
        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        if self.use_residual:
            return x + self.conv(x)
        return self.conv(x)

block = InvertedResidual(32, 16, stride=1, expand_ratio=6)
x = torch.randn(1, 32, 112, 112)
print(block(x).shape)  # (1, 16, 112, 112)

### 5.3 FPN (Feature Pyramid Network)

**核心思想：** 自顶向下融合多尺度特征，构建具有**丰富语义信息**的金字塔特征层次。

**网络结构：**
1. **Bottom-up**（自底向上）：骨干网络（如 ResNet）的前向传播，产生 C2~C5（尺寸递减，语义递增）
2. **Top-down**（自顶向下）：高层特征上采样（`F.interpolate` nearest）后与底层特征相加
3. **Lateral connection**（横向连接）：1×1 卷积统一通道数，使不同层可以直接相加
4. **Smooth**（平滑）：3×3 卷积消除上采样带来的混叠效应

**面试要点：**
1. **为什么需要 FPN？** 小目标需要高分辨率特征（底层），大目标需要强语义特征（高层），FPN 让每层都有强语义
2. **上采样为什么用 nearest 而非 bilinear？** 计算更快，且特征图上采样不需要平滑插值
3. **P2~P5 的用途**：不同层负责不同尺度目标的检测，P2 检测小目标，P5 检测大目标

**注意事项：**
- FPN 是目标检测（如 Faster R-CNN、RetinaNet）和实例分割（如 Mask R-CNN）的基础组件
- `laterals[i] + upsampled` 是逐元素相加，不是拼接（通道数已统一）
- 输出 [P2, P3, P4, P5] 的空间尺寸分别是输入的 1/4, 1/8, 1/16, 1/32

In [ ]:
class FPN(nn.Module):
    def __init__(self, in_channels_list, out_channels=256):
        super().__init__()
        self.lateral_convs = nn.ModuleList()
        self.smooth_convs = nn.ModuleList()
        for in_channels in in_channels_list:
            self.lateral_convs.append(nn.Conv2d(in_channels, out_channels, 1))
            self.smooth_convs.append(nn.Conv2d(out_channels, out_channels, 3, padding=1))

    def forward(self, features):
        """features: list of [C2, C3, C4, C5] 从低到高"""
        laterals = [conv(f) for conv, f in zip(self.lateral_convs, features)]
        
        # 自顶向下融合
        for i in range(len(laterals) - 2, -1, -1):
            upsampled = F.interpolate(laterals[i + 1], size=laterals[i].shape[2:], mode='nearest')
            laterals[i] = laterals[i] + upsampled
        
        # 平滑
        outputs = [smooth(lat) for smooth, lat in zip(self.smooth_convs, laterals)]
        return outputs  # [P2, P3, P4, P5]

# 模拟 backbone 输出
c2 = torch.randn(1, 256, 64, 64)
c3 = torch.randn(1, 512, 32, 32)
c4 = torch.randn(1, 1024, 16, 16)
c5 = torch.randn(1, 2048, 8, 8)

fpn = FPN([256, 512, 1024, 2048], out_channels=256)
p2, p3, p4, p5 = fpn([c2, c3, c4, c5])
print(f"P2: {p2.shape}, P3: {p3.shape}, P4: {p4.shape}, P5: {p5.shape}")

---
## 6. 损失函数手写

### 6.1 Cross Entropy Loss（手写版）

**核心公式：** $L = -\frac{1}{N}\sum_i \log p_{y_i}$，其中 $p_{y_i}$ 是模型对真实类别的预测概率。

**数值稳定实现步骤：**
1. `log-softmax`：先减最大值，再计算 `log(sum(exp(x))) + max`（避免溢出）
2. `log_probs = logits - log_sum_exp`：这就是 log-softmax
3. 用 `log_probs[range(N), targets]` 通过 fancy indexing 取出正确类别的 log 概率

**面试要点：**
1. **为什么不用 softmax + log 分两步？** softmax 可能产生接近 0 的值，`log(0) = -inf`。`log_softmax` 在数学上等价但数值更稳定
2. **Cross Entropy = Negative Log Likelihood (NLL) + Softmax**：PyTorch 中 `F.cross_entropy` 内置了 softmax，输入是 logits
3. `F.nll_loss` 需要输入已经过 `log_softmax` 的概率

**注意事项：**
- 输入 `logits` 是**未经 softmax 的原始输出**，不要先 softmax 再传入
- `range(len(targets))` 生成 batch 索引，配合 `targets` 形成 (batch_idx, class_idx) 的二维索引
- 手写版与 `F.cross_entropy` 的结果应完全一致（差异 < 1e-6）

In [ ]:
def cross_entropy_loss(logits, targets):
    """
    logits: (N, C) 未经 softmax
    targets: (N,) 整数标签
    """
    # log-softmax (数值稳定)
    max_logits = logits.max(dim=-1, keepdim=True).values
    log_sum_exp = (logits - max_logits).exp().sum(dim=-1, keepdim=True).log() + max_logits
    log_probs = logits - log_sum_exp
    
    # gather targets
    loss = -log_probs[range(len(targets)), targets]
    return loss.mean()

# 验证
logits = torch.randn(4, 10)
targets = torch.tensor([1, 3, 5, 7])

my_loss = cross_entropy_loss(logits, targets)
ref_loss = F.cross_entropy(logits, targets)
print(f"手写: {my_loss.item():.6f}")
print(f"官方: {ref_loss.item():.6f}")
print(f"差值: {abs(my_loss - ref_loss).item():.8f}")

### 6.2 Focal Loss

**核心公式：** $FL = -\alpha (1 - p_t)^\gamma \log(p_t)$

- $p_t$：模型对正确类别的预测概率（越接近 1 表示越"容易"）
- $\gamma$（聚焦参数）：γ=0 时退化为标准 CE，γ 越大对困难样本的聚焦越强
- $\alpha$（平衡因子）：调节正负样本的权重

**设计动机（RetinaNet 论文）：** 目标检测中正负样本极度不平衡（大量易分背景），Focal Loss 降低容易样本的 loss 权重，让模型聚焦于困难样本。

**面试要点：**
1. **$(1-p_t)^\gamma$ 的作用**：当样本容易（$p_t$ 接近 1），$(1-p_t)^\gamma$ 接近 0，loss 被大幅抑制；当样本困难（$p_t$ 小），该项接近 1，loss 权重基本不变
2. **γ 的典型值**：2.0（论文推荐），α 的典型值：0.25
3. **与 OHEM 的区别**：OHEM 硬性丢弃容易样本，Focal Loss 软性降低权重（可微分，训练更稳定）

**注意事项：**
- `pt = torch.exp(-ce_loss)` 等价于 `pt = softmax(logits)[targets]`（利用了 `exp(-(-log(p))) = p`）
- Focal Loss 主要用于目标检测，在分类任务中如果类别不平衡也可以使用

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce_loss)  # 正确类别的概率
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        return focal_loss

logits = torch.randn(8, 5)
targets = torch.tensor([0, 1, 2, 3, 4, 0, 1, 2])
loss_fn = FocalLoss(alpha=0.25, gamma=2.0)
print(f"Focal Loss: {loss_fn(logits, targets).item():.4f}")

### 6.3 Smooth L1 Loss (Huber Loss)

**核心公式：**
$$\text{loss}(x) = \begin{cases} 0.5 x^2 / \beta & \text{if } |x| < \beta \\ |x| - 0.5\beta & \text{otherwise} \end{cases}$$

**设计动机：**
- L1 Loss：对异常值鲁棒（梯度恒定），但在零点附近不光滑（优化不稳定）
- L2 Loss：零点附近光滑，但对异常值敏感（梯度随误差线性增长）
- **Smooth L1 = L1 和 L2 的结合**：小误差时用 L2（光滑），大误差时用 L1（鲁棒）

**面试要点：**
1. **β 控制分界点**：β=1.0 是 Fast R-CNN 中的默认值，β 越大越接近 L2
2. **在目标检测中的用途**：边界框回归（bbox regression），预测框与 GT 框的偏移量
3. **梯度特性**：
   - 当 |x| < β：梯度 = x/β（随误差变化，收敛快）
   - 当 |x| ≥ β：梯度 = ±1（恒定，不会因异常值梯度爆炸）

**注意事项：**
- PyTorch 的 `F.huber_loss(delta=β)` 等价于此实现
- Faster R-CNN 和 YOLO 系列都使用 Smooth L1 或其变体做 bbox 回归损失
- 当 β → 0 时趋近 L1，β → ∞ 时趋近 L2

In [ ]:
def smooth_l1_loss(pred, target, beta=1.0):
    diff = torch.abs(pred - target)
    loss = torch.where(diff < beta, 0.5 * diff ** 2 / beta, diff - 0.5 * beta)
    return loss.mean()

pred = torch.randn(16, 4)  # e.g. bbox 回归
target = torch.randn(16, 4)
print(f"手写 Smooth L1: {smooth_l1_loss(pred, target).item():.4f}")
print(f"官方 Huber:     {F.huber_loss(pred, target, delta=1.0).item():.4f}")

### 6.4 Dice Loss（分割常用）

**核心公式：** $Dice = \frac{2|P \cap G| + \epsilon}{|P| + |G| + \epsilon}$，$DiceLoss = 1 - Dice$

- Dice 系数衡量两个集合的重叠程度，范围 [0, 1]（1 表示完全重叠）
- $\epsilon$（smooth）：防止除零，同时起到平滑梯度的作用

**面试要点：**
1. **Dice Loss vs CE Loss**：
   - CE Loss 基于像素级别优化，**对小目标不敏感**（大量背景像素主导）
   - Dice Loss 基于**集合级别**优化，对小目标和类别不平衡更鲁棒
2. **实际中常组合使用**：`Total Loss = CE Loss + λ × Dice Loss`（λ 通常为 0.5~1.0）
3. **one_hot 编码**：将整数标签转为 one-hot 格式，才能与概率图做逐元素运算

**注意事项：**
- Dice Loss 的梯度在训练初期可能不稳定（预测接近全 0 或全 1 时），通常需要先 warm up
- `permute(0, 3, 1, 2)` 将 one-hot 的通道维度从最后移到第二维（NHWC → NCHW）
- `smooth` 参数默认 1e-6，过大会影响 loss 的灵敏度

In [ ]:
def dice_loss(pred, target, smooth=1e-6):
    """
    pred: (N, C, H, W) 经过 softmax 的概率
    target: (N, H, W) 整数标签
    """
    num_classes = pred.shape[1]
    target_onehot = F.one_hot(target, num_classes).permute(0, 3, 1, 2).float()
    
    intersection = (pred * target_onehot).sum(dim=(2, 3))
    union = pred.sum(dim=(2, 3)) + target_onehot.sum(dim=(2, 3))
    
    dice = (2 * intersection + smooth) / (union + smooth)
    return 1 - dice.mean()

pred = torch.softmax(torch.randn(2, 3, 64, 64), dim=1)
target = torch.randint(0, 3, (2, 64, 64))
print(f"Dice Loss: {dice_loss(pred, target).item():.4f}")

---
## 7. 目标检测组件

### 7.1 IoU (Intersection over Union)

**核心公式：** $IoU = \frac{|A \cap B|}{|A \cup B|}$

**计算步骤：**
1. 计算交集矩形的坐标：`inter_x1 = max(x1_a, x1_b)`, `inter_x2 = min(x2_a, x2_b)`
2. 交集面积：宽和高用 `clamp(min=0)` 处理不相交的情况
3. 并集面积 = A 面积 + B 面积 - 交集面积

**面试要点：**
1. **支持广播**：`box1 (N,4)` 与 `box2 (M,4)` 通过 `unsqueeze` 扩展为 `(N,1,4)` 和 `(1,M,4)`，输出 `(N,M)` 的 IoU 矩阵
2. **坐标格式**：`(x1, y1, x2, y2)` 左上角和右下角，注意与 `(cx, cy, w, h)` 中心格式的转换
3. IoU 范围 [0, 1]：0 表示完全不重叠，1 表示完全重合

**注意事项：**
- `clamp(min=0)` 非常关键：当两框不相交时，`inter_x2 - inter_x1` 可能为负，clamp 确保面积为 0
- 分母加 `1e-6` 防止除零（两框面积都为 0 时）
- IoU 是可微的（参与梯度计算），但最大值操作不可导

In [ ]:
def compute_iou(box1, box2):
    """
    box: (x1, y1, x2, y2) 格式
    支持广播: box1 (N, 4), box2 (M, 4) -> (N, M)
    """
    box1 = box1.unsqueeze(1)  # (N, 1, 4)
    box2 = box2.unsqueeze(0)  # (1, M, 4)
    
    inter_x1 = torch.max(box1[..., 0], box2[..., 0])
    inter_y1 = torch.max(box1[..., 1], box2[..., 1])
    inter_x2 = torch.min(box1[..., 2], box2[..., 2])
    inter_y2 = torch.min(box1[..., 3], box2[..., 3])
    
    inter_area = (inter_x2 - inter_x1).clamp(min=0) * (inter_y2 - inter_y1).clamp(min=0)
    
    area1 = (box1[..., 2] - box1[..., 0]) * (box1[..., 3] - box1[..., 1])
    area2 = (box2[..., 2] - box2[..., 0]) * (box2[..., 3] - box2[..., 1])
    
    return inter_area / (area1 + area2 - inter_area + 1e-6)

# 测试
boxes1 = torch.tensor([[0, 0, 10, 10], [5, 5, 15, 15]], dtype=torch.float32)
boxes2 = torch.tensor([[0, 0, 10, 10], [8, 8, 18, 18]], dtype=torch.float32)
iou = compute_iou(boxes1, boxes2)
print(iou)
print("完全重叠 IoU:", iou[0, 0].item())
print("部分重叠 IoU:", iou[0, 1].item())

### 7.2 NMS (Non-Maximum Suppression)

**核心流程：**
1. 按置信度**降序排列**所有检测框
2. 取最高分的框加入结果集，计算它与其余所有框的 IoU
3. **删除** IoU > 阈值的框（认为是重复检测）
4. 重复 2-3 直到所有框处理完毕

**面试要点：**
1. **为什么需要 NMS？** 同一目标可能被多次检测（多个高置信度框重叠），NMS 去除冗余
2. **IoU 阈值的选择**：通常 0.5~0.7。过低会误删不同目标的框（密集场景），过高会保留冗余框
3. **Soft-NMS**：不直接删除，而是降低重叠框的置信度（如乘以 `1 - IoU`），对密集目标效果更好
4. **时间复杂度**：O(N²) 最坏情况，实际中因为框快速减少而远好于此

**注意事项：**
- NMS 是**后处理步骤**，不参与梯度计算（不可微）
- YOLO、Faster R-CNN、SSD 等检测器都使用 NMS
- 代码中 `order[1:]` 取除第一个外的所有索引，`mask = iou <= iou_threshold` 过滤重叠框

In [ ]:
def nms(boxes, scores, iou_threshold=0.5):
    """
    boxes: (N, 4) xyxy
    scores: (N,)
    返回保留的索引
    """
    order = scores.argsort(descending=True)
    keep = []
    
    while order.numel() > 0:
        i = order[0].item()
        keep.append(i)
        
        if order.numel() == 1:
            break
        
        rest = order[1:]
        iou = compute_iou(boxes[i].unsqueeze(0), boxes[rest]).squeeze(0)
        mask = iou <= iou_threshold
        order = rest[mask]
    
    return torch.tensor(keep, dtype=torch.long)

# 模拟检测结果
boxes = torch.tensor([
    [10, 10, 50, 50],
    [12, 12, 52, 52],  # 与第0个高度重叠
    [100, 100, 150, 150],
    [105, 105, 155, 155],  # 与第2个高度重叠
], dtype=torch.float32)
scores = torch.tensor([0.9, 0.8, 0.7, 0.6])

keep = nms(boxes, scores, iou_threshold=0.5)
print("NMS 保留索引:", keep)
print("保留的框:", boxes[keep])

### 7.3 Anchor 生成（YOLO/Faster R-CNN 风格）

**核心概念：** Anchor（先验框）是预定义的、不同尺度和长宽比的参考框，模型预测**相对于 anchor 的偏移量**而非绝对坐标。

**生成步骤：**
1. 遍历特征图的每个空间位置 (y, x)
2. 将特征图坐标映射回输入图像：`cx = (x + 0.5) * stride`
3. 对每个位置，生成 `scales × ratios` 个不同大小和比例的 anchor

**面试要点：**
1. **Anchor 的意义**：直接回归绝对坐标很难学习（目标尺度变化大），回归偏移量更容易优化
2. **Anchor 总数** = H × W × (num_scales × num_ratios)，如 13×13×6 = 1014 个
3. **YOLO vs Faster R-CNN**：
   - YOLOv3：3 个 scale × 3 个 ratio = 9 个 anchor/位置
   - Faster R-CNN：通常也是 9 个/位置（3 scale × 3 ratio）
4. **Anchor-free 趋势**：FCOS、CenterNet 等不使用 anchor，直接预测中心点或到边界的距离

**注意事项：**
- `(x + 0.5)` 是因为特征图像素中心对齐，不是角点对齐
- 实际项目中 anchor 的 scales/ratios 通常通过 K-Means 聚类数据集 GT 框得到（见 12.3）
- stride 等于骨干网络的总下采样倍数（如 32）

In [ ]:
def generate_anchors(feature_size, stride, scales, ratios):
    """
    生成 anchor boxes
    feature_size: (H, W) 特征图大小
    stride: 下采样倍数
    scales: [s1, s2, ...]
    ratios: [(r1_h, r1_w), ...]
    """
    h, w = feature_size
    anchors = []
    for y in range(h):
        for x in range(w):
            cx = (x + 0.5) * stride
            cy = (y + 0.5) * stride
            for scale in scales:
                for ratio_h, ratio_w in ratios:
                    anchor_w = scale * ratio_w
                    anchor_h = scale * ratio_h
                    anchors.append([
                        cx - anchor_w / 2,
                        cy - anchor_h / 2,
                        cx + anchor_w / 2,
                        cy + anchor_h / 2,
                    ])
    return torch.tensor(anchors, dtype=torch.float32)

anchors = generate_anchors(
    feature_size=(13, 13), stride=32,
    scales=[10, 20],
    ratios=[(1, 1), (1.5, 0.67), (0.67, 1.5)]
)
print(f"Anchor 总数: {anchors.shape}")
print(f"前3个: {anchors[:3]}")

### 7.4 GIoU / CIoU

**GIoU 公式：** $GIoU = IoU - \frac{|C - (A \cup B)|}{|C|}$

- $C$：A 和 B 的最小外接矩形（enclosing box）
- GIoU 范围 [-1, 1]，当两框不相交时 IoU=0 但 GIoU < 0（能反映距离）

**GIoU vs IoU 的优势：**
1. IoU=0 时梯度也为 0，无法优化。GIoU 在不相交时仍有梯度
2. GIoU 考虑了非重叠区域的大小，能引导框向正确方向移动

**面试要点：**
1. **IoU 演进路线**：IoU → GIoU → DIoU → CIoU
   - **DIoU**：考虑中心点距离 + 覆盖面积，收敛更快
   - **CIoU**：在 DIoU 基础上加上长宽比一致性，最完整的 IoU 指标
2. YOLOv5/v7/v8 使用 **CIoU Loss** 作为 bbox 回归损失
3. GIoU Loss = 1 - GIoU，作为训练损失使用

**注意事项：**
- GIoU 需要计算最小外接矩形，额外开销比 IoU 稍大
- 当一个框完全包含另一个框时，GIoU 退化为 IoU（外接矩形 = 大框）
- 实际训练中通常直接用 CIoU（综合性能最好）

In [ ]:
def compute_giou(box1, box2):
    """GIoU: box1 (N,4), box2 (N,4) xyxy 格式"""
    # 交集
    inter_x1 = torch.max(box1[:, 0], box2[:, 0])
    inter_y1 = torch.max(box1[:, 1], box2[:, 1])
    inter_x2 = torch.min(box1[:, 2], box2[:, 2])
    inter_y2 = torch.min(box1[:, 3], box2[:, 3])
    inter = (inter_x2 - inter_x1).clamp(min=0) * (inter_y2 - inter_y1).clamp(min=0)
    
    area1 = (box1[:, 2] - box1[:, 0]) * (box1[:, 3] - box1[:, 1])
    area2 = (box2[:, 2] - box2[:, 0]) * (box2[:, 3] - box2[:, 1])
    union = area1 + area2 - inter
    
    # 最小外接矩形
    enc_x1 = torch.min(box1[:, 0], box2[:, 0])
    enc_y1 = torch.min(box1[:, 1], box2[:, 1])
    enc_x2 = torch.max(box1[:, 2], box2[:, 2])
    enc_y2 = torch.max(box1[:, 3], box2[:, 3])
    enc_area = (enc_x2 - enc_x1) * (enc_y2 - enc_y1)
    
    iou = inter / (union + 1e-6)
    giou = iou - (enc_area - union) / (enc_area + 1e-6)
    return giou

box1 = torch.tensor([[0, 0, 10, 10]], dtype=torch.float32)
box2 = torch.tensor([[5, 5, 15, 15]], dtype=torch.float32)
print(f"GIoU: {compute_giou(box1, box2).item():.4f}")

---
## 8. 注意力机制

### 8.1 Squeeze-and-Excitation (SE) Block

**核心思想：** 通道注意力——自适应地学习每个通道的重要性权重，"关注"有用的通道，"抑制"无用的通道。

**两个操作：**
1. **Squeeze（压缩）**：`AdaptiveAvgPool2d(1)` 将每个通道压缩为单个数值（全局信息）
2. **Excitation（激励）**：两层 FC（降维 → ReLU → 升维 → Sigmoid）生成每个通道的权重

**数据流：**
- 输入 (B, C, H, W) → Pool → (B, C) → FC → (B, C//r) → ReLU → FC → (B, C) → Sigmoid → (B, C, 1, 1)
- 最终：`x * y` 逐通道缩放原始特征

**面试要点：**
1. **reduction ratio**（压缩比）：通常为 16，控制中间层的通道数，平衡表达力和计算量
2. SE Block 可以嵌入到任何网络中（ResNet+SE = SE-ResNet，MobileNet+SE = SE-MobileNet）
3. 计算开销很小（只有两个 FC 层），但通常能提升 1~2% 的精度

**注意事项：**
- `view(b, c, 1, 1)` 将权重reshape 为 (B, C, 1, 1) 以便与 (B, C, H, W) 广播相乘
- SE 是**通道注意力**，只关注"哪些通道重要"，不关注空间位置

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        y = self.pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y

se = SEBlock(64)
x = torch.randn(2, 64, 32, 32)
print(se(x).shape)  # (2, 64, 32, 32)

### 8.2 CBAM (Channel + Spatial Attention)

**核心思想：** 结合**通道注意力**和**空间注意力**，同时关注"关注哪些通道"和"关注哪些位置"。

**两个子模块：**
1. **Channel Attention**：
   - 同时使用 AvgPool 和 MaxPool（互补信息）
   - 共享一个 MLP（用 1×1 Conv 实现），两个 pooling 结果相加后 Sigmoid
2. **Spatial Attention**：
   - 沿通道维度做 AvgPool 和 MaxPool → 拼接为 2 通道
   - 7×7 Conv → Sigmoid，输出空间注意力权重图

**CBAM 流程：** `x → × ChannelAttention → × SpatialAttention → output`

**面试要点：**
1. **为什么 Channel Attention 同时用 Avg 和 Max？** Avg 关注全局统计，Max 关注显著特征，两者互补
2. **Spatial Attention 的 kernel_size=7**：较大的感受野能更好地捕捉空间关系
3. CBAM = SE (通道) + 空间注意力，比单独的 SE 效果更好

**注意事项：**
- Channel Attention 用 Conv2d 而非 Linear（等价但更方便处理 (B, C, 1, 1) 的输入）
- 注意力是**串行**应用的：先通道再空间（论文验证了这个顺序最优）
- CBAM 可插入 ResNet 的 Bottleneck 中（在残差加法之前）

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, mid, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, channels, 1, bias=False),
        )

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return torch.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)

    def forward(self, x):
        avg_out = x.mean(dim=1, keepdim=True)
        max_out = x.max(dim=1, keepdim=True).values
        combined = torch.cat([avg_out, max_out], dim=1)
        return torch.sigmoid(self.conv(combined))

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention()

    def forward(self, x):
        x = x * self.ca(x)
        x = x * self.sa(x)
        return x

cbam = CBAM(64)
x = torch.randn(2, 64, 32, 32)
print(cbam(x).shape)

### 8.3 Multi-Head Self-Attention (ViT 核心)

**核心公式：** $\text{Attention}(Q, K, V) = \text{softmax}(\frac{QK^T}{\sqrt{d_k}}) V$

**Multi-Head 的意义：** 将 QKV 分成多个头，每个头独立计算注意力，让模型同时关注不同子空间的信息。

**数据流：**
1. 输入 (B, N, C) → 线性投影为 QKV → reshape 为 (B, heads, N, head_dim)
2. `Q @ K^T` → 缩放 → softmax → 得到注意力权重 (B, heads, N, N)
3. 注意力权重 × V → 拼接所有头 → 线性投影 → 输出

**面试要点：**
1. **缩放因子 $\sqrt{d_k}$**：当维度大时，点积结果方差大，softmax 容易进入饱和区（梯度接近 0），除以 $\sqrt{d_k}$ 缓解
2. **Self-Attention vs Cross-Attention**：Self-Attention 的 QKV 来自同一输入；Cross-Attention 的 Q 来自一个输入，KV 来自另一个
3. **计算复杂度**：O(N² × d)，N 是序列长度——这是 Transformer 处理大图像时效率瓶颈的根源

**注意事项：**
- `head_dim = embed_dim // num_heads`，需确保 embed_dim 能被 num_heads 整除
- `qkv.permute(2, 0, 3, 1, 4)` 将形状从 (B, N, 3, heads, d) 重排为 (3, B, heads, N, d)
- ViT 中 N = num_patches + 1（多了 1 个 CLS token）

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B, heads, N, head_dim)
        q, k, v = qkv.unbind(0)
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.dropout(attn)
        
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        out = self.proj(out)
        return out

mhsa = MultiHeadSelfAttention(embed_dim=768, num_heads=12)
# 模拟 ViT: 196 个 patch + 1 个 cls token
x = torch.randn(1, 197, 768)
print(mhsa(x).shape)  # (1, 197, 768)

### 8.4 Deformable Attention 简化版

**核心思想（Deformable DETR）：** 不对所有空间位置做全局注意力，而是**预测少量采样点的偏移量**，只在这些偏移位置做注意力计算。

**与标准 Attention 的区别：**
- 标准 Attention：每个 query 关注**所有** key/value → O(N²) 复杂度
- Deformable Attention：每个 query 只关注 **K 个采样点**（K 通常为 4~8）→ O(N×K) 复杂度

**关键组件：**
1. `offset_proj`：预测每个 head 的每个采样点的 (x, y) 偏移量
2. `attention_weights`：预测每个采样点的注意力权重（替代 softmax）
3. 在偏移位置做双线性插值采样 value，加权求和

**面试要点：**
1. **解决了什么问题**：DETR 收敛慢（全局注意力学习困难），Deformable DETR 通过稀疏采样加速收敛
2. **双线性插值**：采样位置可能不在整数像素上，需要用周围 4 个像素插值
3. 这是 Deformable Convolution 思想在 Attention 上的推广

**注意事项：**
- 这里的代码是**结构示意**，省略了双线性插值的具体实现
- 完整实现需要 `F.grid_sample` 进行可变形采样
- Deformable Attention 在 DETR、Mask2Former 等检测/分割模型中广泛使用

In [ ]:
class DeformableAttentionSimplified(nn.Module):
    """简化版可变形注意力 — 仅展示 offset 采样思路"""
    def __init__(self, embed_dim, num_heads, num_points=4):
        super().__init__()
        self.num_heads = num_heads
        self.num_points = num_points
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        # 预测采样偏移量
        self.offset_proj = nn.Linear(embed_dim, num_heads * num_points * 2)
        # 每个采样点的权重
        self.attention_weights = nn.Linear(embed_dim, num_heads * num_points)

    def forward(self, x, H, W):
        B, N, C = x.shape
        q = self.q_proj(x).reshape(B, N, self.num_heads, self.head_dim)
        v = self.v_proj(x).reshape(B, N, self.num_heads, self.head_dim)
        
        offsets = self.offset_proj(x).reshape(B, N, self.num_heads, self.num_points, 2)
        attn_w = self.attention_weights(x).reshape(B, N, self.num_heads, self.num_points)
        attn_w = F.softmax(attn_w, dim=-1)
        
        # 简化: 用 offset 做 bilinear sample (此处仅展示结构)
        out = self.out_proj(v.mean(dim=2, keepdim=True).expand(-1, -1, self.num_heads, -1).reshape(B, N, C))
        return out

deform_attn = DeformableAttentionSimplified(256, 8, num_points=4)
x = torch.randn(1, 196, 256)
print(deform_attn(x, 14, 14).shape)

---
## 9. 数据增强与预处理

### 9.1 常用图像增强（NumPy 实现）

**核心思想：** 数据增强通过对训练图像做随机变换，扩充训练数据的多样性，提高模型泛化能力。

**四种基础增强：**
1. **Random Horizontal Flip**：50% 概率水平翻转。`img[:, ::-1]` 是 NumPy 切片翻转，`.copy()` 确保内存连续
2. **Random Crop**：随机裁剪子区域。注意裁剪尺寸不能超过原图尺寸
3. **Color Jitter**：随机调整亮度、对比度、饱和度。模拟不同光照条件
4. **Normalize**：`(img / 255 - mean) / std`，将像素值标准化到零均值、单位方差

**面试要点：**
1. **训练时增强，测试时不增强**（只用 Normalize）。训练增强引入随机性，测试需要确定性
2. **ImageNet 标准化参数**：mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
3. **增强顺序**：几何变换（flip/crop）→ 颜色变换（jitter）→ 标准化（normalize）

**注意事项：**
- 翻转时如果使用 BBox 标注，BBox 坐标也需要同步变换
- Color Jitter 中 `128 * (1 - alpha)` 是以灰色为基准调整对比度
- 实际项目中使用 `torchvision.transforms` 或 `albumentations` 库

In [ ]:
def random_horizontal_flip(img, p=0.5):
    if random.random() < p:
        return img[:, ::-1].copy()
    return img

def random_crop(img, crop_size):
    h, w = img.shape[:2]
    ch, cw = crop_size
    top = random.randint(0, h - ch)
    left = random.randint(0, w - cw)
    return img[top:top+ch, left:left+cw]

def color_jitter(img, brightness=0.4, contrast=0.4, saturation=0.4):
    """简易颜色抖动"""
    img = img.astype(np.float32)
    if brightness > 0:
        img += random.uniform(-brightness, brightness) * 255
    if contrast > 0:
        alpha = random.uniform(1 - contrast, 1 + contrast)
        img = img * alpha + 128 * (1 - alpha)
    return np.clip(img, 0, 255).astype(np.uint8)

def normalize(img, mean, std):
    """标准化 (H, W, C) uint8 -> float32"""
    img = img.astype(np.float32) / 255.0
    img = (img - np.array(mean)) / np.array(std)
    return img

# 测试
img = np.random.randint(0, 256, (32, 32, 3), dtype=np.uint8)
img_flipped = random_horizontal_flip(img)
img_cropped = random_crop(img, (28, 28))
img_jittered = color_jitter(img)
img_normalized = normalize(img, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
print(f"原图: {img.shape}")
print(f"裁剪: {img_cropped.shape}")
print(f"标准化后均值: {img_normalized.mean(axis=(0,1)).round(3)}")

### 9.2 Mixup & CutMix

**Mixup：**
- 将两张图和它们的标签做**线性插值**：`x = λ*x1 + (1-λ)*x2`
- λ 从 Beta(α, α) 分布采样（α 通常为 0.4 或 1.0）
- 标签也做同样的混合：`y = λ*y1 + (1-λ)*y2`（需要 one-hot 编码）

**CutMix：**
- 从一张图上**裁剪一个矩形区域**，用另一张图的对应区域替换
- λ 同样从 Beta 分布采样，裁剪面积比 = 1 - λ
- 标签按实际面积比例混合（不是原始 λ）

**面试要点：**
1. **Mixup 的正则化效果**：鼓励模型在样本间做平滑过渡，减少过拟合
2. **CutMix 的优势**：保留了更多空间信息（未被遮挡的区域保持原样），比 Mixup 更适合目标检测
3. **Beta 分布**：α=1 时退化为均匀分布，α<1 时 λ 倾向于接近 0 或 1（轻微混合）

**注意事项：**
- CutMix 中实际面积比可能与采样的 λ 不同，需要重新计算 `lam`
- 训练时标签混合意味着使用**软标签**而非硬标签
- 两种方法通常以一定概率应用（如 50% 的 batch 使用 Mixup/CutMix）

In [ ]:
def mixup(x1, y1, x2, y2, alpha=0.4):
    """Mixup: 线性插值混合"""
    lam = np.random.beta(alpha, alpha)
    x = lam * x1 + (1 - lam) * x2
    return x, lam * y1 + (1 - lam) * y2, lam

def cutmix(x1, y1, x2, y2, alpha=1.0):
    """CutMix: 裁剪区域替换"""
    lam = np.random.beta(alpha, alpha)
    H, W = x1.shape[-2], x1.shape[-1]
    cut_ratio = np.sqrt(1 - lam)
    rw, rh = int(W * cut_ratio), int(H * cut_ratio)
    
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1s = max(cx - rw // 2, 0)
    y1s = max(cy - rh // 2, 0)
    x1e = min(cx + rw // 2, W)
    y1e = min(cy + rh // 2, H)
    
    x = x1.clone()
    x[:, :, y1s:y1e, x1s:x1e] = x2[:, :, y1s:y1e, x1s:x1e]
    # 按面积比调整 lambda
    lam = 1 - (x1e - x1s) * (y1e - y1s) / (H * W)
    return x, lam * y1 + (1 - lam) * y2, lam

# 测试
x1, x2 = torch.randn(1, 3, 32, 32), torch.randn(1, 3, 32, 32)
y1 = F.one_hot(torch.tensor(2), 10).float().unsqueeze(0)
y2 = F.one_hot(torch.tensor(5), 10).float().unsqueeze(0)

x_mix, y_mix, lam = mixup(x1, y1, x2, y2)
print(f"Mixup lambda: {lam:.3f}")

x_cut, y_cut, lam = cutmix(x1, y1, x2, y2)
print(f"CutMix lambda: {lam:.3f}")

### 9.3 Mosaic 增强（YOLO 系列）

**核心思想：** 将 4 张图拼成一张大图，再随机裁剪出目标尺寸。等价于一次性看到 4 张图的内容。

**操作步骤：**
1. 创建 2×target_size 的大画布
2. 随机选择拼接中心点 (cx, cy)
3. 将 4 张图分别放在左上、右上、左下、右下四个区域
4. 从大画布中随机裁剪 target_size × target_size 的区域

**面试要点：**
1. **YOLOv4 首次提出**，YOLOv5/v7/v8 都在使用，是 YOLO 系列的标配增强
2. **优势**：
   - 一次看到 4 张图 → 等效 batch size × 4
   - 小目标更多（图被缩小后拼接）→ 提升小目标检测
   - 背景更丰富 → 减少对特定背景的过拟合
3. **训练后期关闭**：最后 10~15 个 epoch 关闭 Mosaic，用正常图片 fine-tune

**注意事项：**
- Mosaic 后目标的 BBox 坐标需要相应变换（不能直接用原图标注）
- 实际实现中需要处理目标被裁剪到画布外的情况（截断 BBox）
- Mosaic 通常与 Mixup/CutMix 叠加使用（YOLOv5 的默认策略）

In [ ]:
def mosaic_augment(images, target_size=416):
    """将4张图拼成一张 (YOLO 常用)"""
    mosaic = np.zeros((target_size * 2, target_size * 2, 3), dtype=np.float32)
    
    # 随机选择拼接中心点
    cx = np.random.randint(target_size // 2, target_size * 3 // 2)
    cy = np.random.randint(target_size // 2, target_size * 3 // 2)
    
    positions = [
        (0, 0, cx, cy),             # 左上
        (cx, 0, target_size * 2, cy),  # 右上
        (0, cy, cx, target_size * 2),  # 左下
        (cx, cy, target_size * 2, target_size * 2),  # 右下
    ]
    
    for img, (x1, y1, x2, y2) in zip(images, positions):
        # resize 到目标区域
        h, w = y2 - y1, x2 - x1
        # 简单 resize (实际用 cv2.resize)
        resized = np.array(img[::max(1, img.shape[0]//h), ::max(1, img.shape[1]//w)][:h, :w])
        rh, rw = resized.shape[:2]
        mosaic[y1:y1+rh, x1:x1+rw] = resized
    
    # 裁剪中心 target_size x target_size
    cx2 = np.random.randint(target_size // 2, target_size * 3 // 2)
    cy2 = np.random.randint(target_size // 2, target_size * 3 // 2)
    x1 = max(cx2 - target_size // 2, 0)
    y1 = max(cy2 - target_size // 2, 0)
    return mosaic[y1:y1+target_size, x1:x1+target_size]

imgs = [np.random.randint(0, 256, (300, 300, 3), dtype=np.uint8) for _ in range(4)]
result = mosaic_augment(imgs, target_size=128)
print(f"Mosaic 输出: {result.shape}")

---
## 10. 评价指标

### 10.1 混淆矩阵 & Precision / Recall / F1

**混淆矩阵**：行 = 真实标签，列 = 预测标签。矩阵元素 `mat[i][j]` 表示真实类别 i 被预测为 j 的数量。

**核心指标（每个类别）：**
- **Precision（精确率）** = TP / (TP + FP) = "预测为正的中有多少是对的"
- **Recall（召回率）** = TP / (TP + FN) = "真实为正的有多少被找到了"
- **F1** = 2 × P × R / (P + R) = Precision 和 Recall 的调和平均

**面试要点：**
1. **Precision vs Recall 的权衡**：
   - 提高阈值 → Precision 升高、Recall 降低（更保守，少预测但准）
   - 降低阈值 → Precision 降低、Recall 升高（更激进，多预测但不准）
2. **多分类的 F1**：
   - Macro-F1：各类 F1 的简单平均（每类权重相同）
   - Micro-F1：先汇总所有 TP/FP/FN 再计算（受大类影响）
   - Weighted-F1：按各类样本数加权平均
3. **混淆矩阵对角线**元素越大越好（正确预测），非对角线表示误分类

**注意事项：**
- 分母为 0 时需特殊处理（如 TP+FP=0 时 Precision 定义为 0）
- 目标检测中更关注 AP/mAP，分类任务更关注 F1

In [ ]:
def confusion_matrix(y_true, y_pred, num_classes):
    mat = np.zeros((num_classes, num_classes), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        mat[t, p] += 1
    return mat

def precision_recall_f1(y_true, y_pred, num_classes):
    mat = confusion_matrix(y_true, y_pred, num_classes)
    results = {}
    for c in range(num_classes):
        tp = mat[c, c]
        fp = mat[:, c].sum() - tp
        fn = mat[c, :].sum() - tp
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        results[c] = {'precision': precision, 'recall': recall, 'f1': f1}
    return results

y_true = [0, 1, 2, 0, 1, 2, 0, 1, 2]
y_pred = [0, 1, 1, 0, 1, 2, 2, 1, 2]
results = precision_recall_f1(y_true, y_pred, 3)
for c, m in results.items():
    print(f"Class {c}: P={m['precision']:.2f} R={m['recall']:.2f} F1={m['f1']:.2f}")

### 10.2 mAP 计算 (AP @ IoU=0.5)

**核心概念：** AP (Average Precision) = Precision-Recall 曲线下的面积。mAP = 所有类别 AP 的平均值。

**计算流程：**
1. **对每个类别**：收集所有检测结果，按置信度降序排列
2. 逐个判定 TP/FP：与 GT 的 IoU ≥ 阈值且未被匹配 → TP，否则 → FP
3. 计算累积的 Precision 和 Recall → 绘制 P-R 曲线
4. AP = P-R 曲线下的面积（用梯形法数值积分）

**面试要点：**
1. **COCO mAP vs VOC mAP**：
   - VOC: mAP@0.5（IoU 阈值固定为 0.5）
   - COCO: mAP@[0.5:0.95]（IoU 阈值从 0.5 到 0.95 每 0.05 取一次，求平均）
2. **AP 计算中的平滑处理**：`mpre[i] = max(mpre[i], mpre[i+1])` 确保 Precision 单调递减，消除锯齿波动
3. **mAP 是目标检测最核心的指标**，面试必考

**注意事项：**
- 一个 GT 只能匹配一个检测框（一对一），重复匹配算 FP
- 如果某类没有 GT，该类的 AP 不计入 mAP（跳过）
- `compute_iou` 在 7.1 中定义，这里复用了该函数

In [ ]:
def compute_ap(recall, precision):
    """计算单类 AP (11点插值 或 全点)"""
    # 添加首尾点
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    
    # 确保 precision 单调递减
    for i in range(len(mpre) - 2, -1, -1):
        mpre[i] = max(mpre[i], mpre[i + 1])
    
    # 计算面积
    indices = np.where(mrec[1:] != mrec[:-1])[0]
    ap = np.sum((mrec[indices + 1] - mrec[indices]) * mpre[indices + 1])
    return ap

def compute_map_per_class(all_detections, all_groundtruths, iou_threshold=0.5, num_classes=3):
    """
    简化版 mAP 计算
    all_detections: list of (boxes, scores, labels) per image
    all_groundtruths: list of (boxes, labels) per image
    """
    aps = []
    for cls in range(num_classes):
        # 收集该类所有检测结果并按置信度排序
        all_scores = []
        all_tp_fp = []  # 1=TP, 0=FP
        n_gt = 0
        
        for dets, gts in zip(all_detections, all_groundtruths):
            det_boxes, det_scores, det_labels = dets
            gt_boxes, gt_labels = gts
            
            gt_mask = gt_labels == cls
            n_gt += gt_mask.sum()
            gt_cls = gt_boxes[gt_mask]
            
            det_mask = det_labels == cls
            det_cls_boxes = det_boxes[det_mask]
            det_cls_scores = det_scores[det_mask]
            
            if len(det_cls_boxes) == 0:
                continue
            
            sort_idx = det_cls_scores.argsort()[::-1]
            det_cls_boxes = det_cls_boxes[sort_idx]
            det_cls_scores = det_cls_scores[sort_idx]
            
            matched = set()
            for j in range(len(det_cls_boxes)):
                all_scores.append(det_cls_scores[j])
                if len(gt_cls) == 0:
                    all_tp_fp.append(0)
                    continue
                ious = compute_iou(det_cls_boxes[j:j+1], gt_cls).squeeze(0)
                best_idx = ious.argmax().item()
                best_iou = ious[best_idx].item()
                if best_iou >= iou_threshold and best_idx not in matched:
                    all_tp_fp.append(1)
                    matched.add(best_idx)
                else:
                    all_tp_fp.append(0)
        
        if n_gt == 0:
            continue
        
        tp_fp = np.array(all_tp_fp)
        tp_cumsum = np.cumsum(tp_fp)
        fp_cumsum = np.cumsum(1 - tp_fp)
        recall = tp_cumsum / n_gt
        precision = tp_cumsum / (tp_cumsum + fp_cumsum)
        ap = compute_ap(recall, precision)
        aps.append(ap)
        print(f"  Class {cls}: AP = {ap:.4f}")
    
    return np.mean(aps) if aps else 0.0

# 简单测试
dets = [
    (torch.tensor([[10,10,50,50],[60,60,100,100],[20,20,60,60]], dtype=torch.float32),
     torch.tensor([0.9, 0.8, 0.3]),
     torch.tensor([0, 1, 0])),
]
gts = [
    (torch.tensor([[10,10,50,50],[60,60,100,100]], dtype=torch.float32),
     torch.tensor([0, 1])),
]
mAP = compute_map_per_class(dets, gts, iou_threshold=0.5, num_classes=3)
print(f"mAP@0.5: {mAP:.4f}")

### 10.3 分割指标 — mIoU

**核心公式：** $IoU_c = \frac{|P_c \cap G_c|}{|P_c \cup G_c|}$，$mIoU = \frac{1}{C}\sum_c IoU_c$

- 对每个类别计算预测和 GT 的交集与并集之比，再对所有类别求平均

**面试要点：**
1. **mIoU 是语义分割最核心的指标**，几乎所有分割论文都报告 mIoU
2. **与 mAP 的区别**：mAP 针对目标检测（基于 bbox），mIoU 针对分割（基于像素）
3. **实际使用中的细节**：
   - 通常忽略"背景"类或"忽略"标签
   - 空类别（GT 中不存在）不计入平均
4. **其他分割指标**：Pixel Accuracy（像素准确率）、Dice Coefficient（与 IoU 的关系：Dice = 2IoU/(1+IoU)）

**注意事项：**
- `pred_cls & target_cls` 是位运算（布尔与），计算交集像素数
- `pred_cls | target_cls` 是位运算（布尔或），计算并集像素数
- union=0 的类别跳过（GT 中没有该类且模型也没预测该类），不计入平均

In [ ]:
def compute_miou(pred, target, num_classes):
    """
    pred: (N, H, W) 预测标签
    target: (N, H, W) 真实标签
    """
    ious = []
    for cls in range(num_classes):
        pred_cls = (pred == cls)
        target_cls = (target == cls)
        intersection = (pred_cls & target_cls).sum().item()
        union = (pred_cls | target_cls).sum().item()
        if union > 0:
            ious.append(intersection / union)
    return np.mean(ious) if ious else 0.0

pred = torch.randint(0, 3, (2, 64, 64))
target = torch.randint(0, 3, (2, 64, 64))
print(f"mIoU: {compute_miou(pred, target, 3):.4f}")

---
## 11. 训练流程

### 11.1 完整训练模板（CIFAR-10 示例）

**模板包含三个核心部分：**

**1. 模型定义 (SimpleCNN)：**
- 3 个卷积块：Conv → BN → ReLU → Pool
- `AdaptiveAvgPool2d(1)`：无论输入尺寸多大，都输出 1×1（灵活适配）
- 最后一个线性层做分类

**2. train_one_epoch（训练一个 epoch）：**
- `model.train()`：启用 BN 和 Dropout 的训练模式
- 循环：zero_grad → forward → loss → backward → step
- 记录 loss 和 accuracy 用于监控

**3. evaluate（评估/验证）：**
- `@torch.no_grad()`：不计算梯度，节省内存和计算
- `model.eval()`：BN 使用 running stats，Dropout 关闭

**面试要点：**
1. **model.train() vs model.eval()**：影响 BN（用 batch stats 还是 running stats）和 Dropout（随机丢弃还是不丢弃）
2. **loss.item() × images.size(0)**：累加总 loss，最后除以 total 得到平均 loss
3. **训练循环的标准模板**：几乎所有 PyTorch 项目都用这个模式

**注意事项：**
- 验证时一定不要忘记 `model.eval()` 和 `torch.no_grad()`
- `outputs.argmax(1)` 取概率最大的类别作为预测结果

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

# 使用模拟数据快速验证
model = SimpleCNN()
dummy_images = torch.randn(32, 3, 32, 32)
dummy_labels = torch.randint(0, 10, (32,))
out = model(dummy_images)
print(f"输出 shape: {out.shape}")  # (32, 10)

### 11.2 学习率调度器手写

**Cosine Annealing with Linear Warmup** 是目前最常用的学习率调度策略。

**两个阶段：**
1. **Warmup（线性预热）**：学习率从 0 线性增长到 base_lr
   - 目的：训练初期参数随机，大学习率可能导致训练不稳定
   - 通常 warmup_steps = 总步数的 5~10%
2. **Cosine Annealing（余弦退火）**：学习率按余弦曲线从 base_lr 衰减到 min_lr
   - 公式：$lr = \frac{1}{2}(1 + \cos(\pi \cdot progress)) \times base\_lr$
   - 平滑衰减，比 step decay 更稳定

**面试要点：**
1. **为什么需要 warmup？** 深度网络初期梯度不稳定，大学习率可能使参数偏离初始化的合理范围
2. **常见调度器对比**：
   - StepLR：每隔固定 epoch 衰减（简单但不平滑）
   - CosineAnnealing：平滑衰减（当前主流）
   - OneCycleLR：先增后减（fastai 提出，训练快）
   - ReduceLROnPlateau：根据指标自适应调整
3. 大多数 SOTA 模型（ViT、Swin、DETR）都使用 Cosine + Warmup

**注意事项：**
- `scheduler.step()` 应在每个 training step 后调用（不是 epoch 后）
- 保存/恢复训练时需要同时保存 scheduler 的 current_step

In [ ]:
class CosineAnnealingWarmup:
    """Cosine Annealing with Linear Warmup"""
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr=1e-6):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr = min_lr
        self.base_lrs = [pg['lr'] for pg in optimizer.param_groups]
        self.current_step = 0

    def step(self):
        self.current_step += 1
        if self.current_step <= self.warmup_steps:
            scale = self.current_step / self.warmup_steps
        else:
            progress = (self.current_step - self.warmup_steps) / (self.total_steps - self.warmup_steps)
            scale = 0.5 * (1 + math.cos(math.pi * progress))
        
        for pg, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            pg['lr'] = max(self.min_lr, base_lr * scale)

    def get_lr(self):
        return [pg['lr'] for pg in self.optimizer.param_groups]

# 可视化 lr 曲线
import matplotlib.pyplot as plt

model_tmp = nn.Linear(10, 2)
optimizer = torch.optim.SGD(model_tmp.parameters(), lr=0.1)
scheduler = CosineAnnealingWarmup(optimizer, warmup_steps=100, total_steps=1000)

lrs = []
for _ in range(1000):
    scheduler.step()
    lrs.append(scheduler.get_lr()[0])

plt.figure(figsize=(8, 3))
plt.plot(lrs)
plt.xlabel('Step')
plt.ylabel('Learning Rate')
plt.title('Cosine Annealing with Linear Warmup')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 11.3 梯度裁剪 & 混合精度训练模板

**梯度裁剪 (Gradient Clipping)：**
- `clip_grad_norm_(model.parameters(), max_grad_norm)`
- 将梯度的 L2 范数限制在 `max_grad_norm` 以内（通常设为 1.0 或 5.0）
- 防止梯度爆炸（尤其在 RNN/Transformer 中常见）
- 不改变梯度方向，只缩放梯度大小

**混合精度训练 (AMP)：**
- 使用 **float16**（半精度）做前向和反向传播，减少显存占用约 50%，加速 2~3 倍
- 用 **GradScaler** 解决 float16 梯度下溢（小梯度变为 0）的问题：
  - `scaler.scale(loss)`：放大 loss，防止小梯度下溢
  - `scaler.unscale_(optimizer)`：恢复梯度原始大小后再做梯度裁剪
  - `scaler.step()`：如果有 inf/nan 则跳过这次更新

**面试要点：**
1. **AMP 的三个步骤**：autocast（自动混合精度）→ scale → unscale → clip → step
2. **为什么需要 GradScaler？** float16 的最小正值约 6e-8，小于此的梯度会变为 0（underflow），scaler 放大 loss 来避免
3. **梯度裁剪的位置**：必须在 `unscale_` 之后、`step` 之前

**注意事项：**
- `autocast('cuda')` 只在 CUDA 上有效，CPU 训练不需要 AMP
- 梯度裁剪的 `max_norm` 通常设为 1.0（Transformer）或 5.0（RNN）
- `scaler.update()` 会动态调整 scale factor

In [ ]:
def train_with_amp(model, loader, criterion, optimizer, device, max_grad_norm=1.0):
    """混合精度训练 + 梯度裁剪"""
    model.train()
    scaler = torch.amp.GradScaler('cuda')
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        scaler.step(optimizer)
        scaler.update()

print("混合精度训练模板已定义 (需要 CUDA 环境)")

---
## 12. 综合实战

### 12.1 手写 ViT Block

**Vision Transformer (ViT) 的核心组件，将 Transformer 从 NLP 引入计算机视觉。**

**三大组件：**
1. **PatchEmbedding**：将图像切分为 patch 并映射为 embedding
   - 用 `Conv2d(kernel=patch_size, stride=patch_size)` 实现——等价于将每个 patch 线性投影
   - 输入 (B, 3, 224, 224) → 输出 (B, 196, 768)（patch 数 = (224/16)² = 196）

2. **TransformerBlock**：标准 Transformer 编码器块
   - Pre-Norm 结构：`x + attn(norm(x))` 和 `x + mlp(norm(x))`
   - MLP：Linear → GELU → Linear（扩展比通常为 4）
   - 残差连接保证梯度流通

3. **SimpleViT**：完整模型
   - CLS token：可学习的特殊 token，最终用它的输出做分类（汇聚全局信息）
   - Position Embedding：为每个 patch 位置加可学习的位置编码（无位置信息则模型无法区分 patch 位置）

**面试要点：**
1. ViT 的归纳偏置比 CNN 弱（没有平移不变性和局部性），需要更多数据才能超越 CNN
2. `flatten(2).transpose(1,2)` 将空间维度展平为序列维度
3. `x[:, 0]` 取 CLS token 的输出做最终分类

**注意事项：**
- ViT 的计算量主要在 Self-Attention（O(N²)），patch 越小 N 越大计算量越大
- 实际 ViT 还需要 Dropout、Stochastic Depth 等正则化

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)  # (B, embed_dim, H/P, W/P)
        x = x.flatten(2).transpose(1, 2)  # (B, num_patches, embed_dim)
        return x

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadSelfAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * mlp_ratio),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * mlp_ratio, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class SimpleViT(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, num_classes=1000,
                 embed_dim=768, depth=12, num_heads=12, mlp_ratio=4):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, self.patch_embed.num_patches + 1, embed_dim) * 0.02)
        
        self.blocks = nn.Sequential(*[
            TransformerBlock(embed_dim, num_heads, mlp_ratio) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        x = x + self.pos_embed
        x = self.blocks(x)
        x = self.norm(x[:, 0])  # cls token
        return self.head(x)

# 测试 (小模型)
vit = SimpleViT(img_size=32, patch_size=8, embed_dim=192, depth=4, num_heads=3, num_classes=10)
x = torch.randn(2, 3, 32, 32)
print(f"ViT 参数量: {sum(p.numel() for p in vit.parameters()) / 1e6:.2f}M")
print(f"输出: {vit(x).shape}")  # (2, 10)

### 12.2 模型参数量 & FLOPs 估算

**参数量 (Parameters)：** 模型中可学习参数的总数，决定了模型的存储大小和显存占用。
- 参数量 × 4 bytes（float32）≈ 模型大小
- 如 25M 参数 ≈ 100MB

**FLOPs (Floating Point Operations)：** 模型一次前向传播的浮点运算次数，衡量计算复杂度。
- 卷积层 FLOPs = 2 × OH × OW × KH × KW × C_in × C_out（乘加算 2 次运算）

**面试要点：**
1. **常见模型的参数量和 FLOPs**：
   - ResNet-50：~25M 参数，~4.1G FLOPs
   - ViT-Base：~86M 参数，~17.6G FLOPs
   - MobileNetV2：~3.4M 参数，~0.3G FLOPs
2. **FLOPs ≠ 实际推理速度**：还受内存带宽、并行度、硬件优化等因素影响
3. **参数量 vs FLOPs**：大核卷积参数少但 FLOPs 高，1×1 卷积参数少 FLOPs 也低

**注意事项：**
- `p.numel()` 返回 tensor 的元素数量（即该参数的参数量）
- 这里是单层 FLOPs 估算，整个模型的 FLOPs 需要逐层累加
- 精确计算建议使用 `thop`、`fvcore` 或 `ptflops` 库
- `requires_grad=False` 的参数（如 BN 的 running_mean）不计入可训练参数

In [ ]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def estimate_conv_flops(in_channels, out_channels, kernel_size, input_size, stride=1):
    """估算单层卷积 FLOPs"""
    if isinstance(kernel_size, int):
        kernel_size = (kernel_size, kernel_size)
    h, w = input_size
    oh = (h - kernel_size[0]) // stride + 1
    ow = (w - kernel_size[1]) // stride + 1
    # 每个输出元素: kernel_h * kernel_w * in_channels 次乘加
    flops = 2 * oh * ow * kernel_size[0] * kernel_size[1] * in_channels * out_channels
    return flops

# 示例: ResNet 第一个 7x7 卷积
flops = estimate_conv_flops(3, 64, 7, (224, 224), stride=2)
print(f"Conv1 FLOPs: {flops / 1e6:.1f}M")

# 模型参数统计
model = SimpleCNN()
total, trainable = count_parameters(model)
print(f"SimpleCNN: 总参数 {total:,}, 可训练 {trainable:,}")

### 12.3 K-Means 聚类生成 Anchor（YOLO 用）

**核心思想：** 使用 K-Means 聚类在数据集的 GT 框上聚类，自动发现最适合数据集的 anchor 尺寸。

**与传统 K-Means 的区别：**
- 传统 K-Means 用**欧氏距离**，大框比小框产生更大的误差
- 这里使用 **IoU 距离**：`d = 1 - IoU`，与框大小无关，更公平

**算法流程：**
1. 随机选 K 个初始中心（GT 框的宽高）
2. 对每个 GT 框，计算与所有中心的 IoU 距离，分配到最近的中心
3. 用 **median**（而非 mean）更新中心，避免异常值影响
4. 重复直到中心不再变化

**面试要点：**
1. **YOLOv5 的 9 个 anchor** 就是通过 K-Means 在 COCO 数据集上聚类得到的
2. **K 值的选择**：通常通过绘制 "K vs Avg IoU" 曲线，选择拐点处的 K 值
3. 聚类结果按面积排序后，分配给不同尺度的特征图（小 anchor → 大特征图，大 anchor → 小特征图）

**注意事项：**
- 输入 `boxes_wh` 是归一化或绝对像素的宽高，不是坐标
- `min_w = np.minimum(boxes[:, 0:1], centers[:, 0])` 利用广播计算交集（假设中心点对齐）
- median 比 mean 更鲁棒，因为 GT 框可能存在极端离群值

In [ ]:
def kmeans_anchors(boxes_wh, k=9, max_iter=100):
    """
    boxes_wh: (N, 2) 每个框的 (width, height)
    返回 k 个 anchor 的 (w, h)
    """
    n = boxes_wh.shape[0]
    # 随机初始化中心
    indices = np.random.choice(n, k, replace=False)
    centers = boxes_wh[indices].copy()
    
    def iou_distance(boxes, centers):
        """基于 IoU 的距离"""
        min_w = np.minimum(boxes[:, 0:1], centers[:, 0])
        min_h = np.minimum(boxes[:, 1:2], centers[:, 1])
        inter = min_w * min_h
        boxes_area = boxes[:, 0] * boxes[:, 1]
        centers_area = centers[:, 0] * centers[:, 1]
        iou = inter / (boxes_area[:, None] + centers_area - inter + 1e-6)
        return 1 - iou
    
    for _ in range(max_iter):
        dist = iou_distance(boxes_wh, centers)
        labels = dist.argmin(axis=1)
        new_centers = np.zeros_like(centers)
        for i in range(k):
            mask = labels == i
            if mask.sum() > 0:
                new_centers[i] = boxes_wh[mask].median(axis=0)
            else:
                new_centers[i] = centers[i]
        if np.allclose(centers, new_centers):
            break
        centers = new_centers
    
    # 按面积排序
    order = np.prod(centers, axis=1).argsort()
    return centers[order]

# 模拟 GT 框
np.random.seed(42)
boxes_wh = np.concatenate([
    np.random.rand(500, 2) * [30, 30] + [5, 5],    # 小目标
    np.random.rand(300, 2) * [60, 60] + [30, 30],  # 中目标
    np.random.rand(200, 2) * [100, 100] + [60, 60], # 大目标
])
anchors = kmeans_anchors(boxes_wh, k=9)
print("Generated Anchors (w, h):")
for i, (w, h) in enumerate(anchors):
    print(f"  Anchor {i}: ({w:.1f}, {h:.1f})")

### 12.4 模型权重的 Initialization

**核心思想：** 好的初始化能避免梯度消失/爆炸，加速收敛。不同类型的层适合不同的初始化策略。

**三种常用初始化：**
1. **Kaiming Normal（He 初始化）**→ 用于 Conv2d + ReLU
   - 方差 $\sigma^2 = 2/n_{fan\_out}$，使 ReLU 网络每层方差保持稳定
   - `mode='fan_out'` 使用输出连接数（比 fan_in 更稳定）

2. **Xavier Uniform（Glorot 初始化）**→ 用于 Linear 层
   - 方差 $\sigma^2 = 2/(n_{in} + n_{out})$，适用于 tanh/sigmoid 激活
   - 均匀分布范围：$[-\sqrt{6/(n_{in}+n_{out})}, +\sqrt{6/(n_{in}+n_{out})}]$

3. **BN/LN 层**：weight 初始化为 1，bias 初始化为 0

**面试要点：**
1. **为什么初始化很重要？** 全 0 初始化 → 所有神经元输出相同 → 梯度相同 → 对称性无法打破
2. **ReLU 用 Kaiming，tanh/sigmoid 用 Xavier**——因为 ReLU 会"杀死"一半神经元，需要额外补偿
3. `model.apply(fn)` 递归地对所有子模块调用 `fn`，是 PyTorch 初始化的标准写法

**注意事项：**
- 偏置通常初始化为 0（少数例外：YOLO 的置信度偏置初始化为 -log(1/α - 1)）
- Pre-trained 模型不需要初始化（已有训练好的权重）
- `nonlinearity='relu'` 参数确保 Kaiming 使用正确的方差公式

In [ ]:
def init_weights(module):
    """常用的权重初始化"""
    if isinstance(module, nn.Conv2d):
        # Kaiming 初始化 (适合 ReLU)
        nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Linear):
        # Xavier 初始化
        nn.init.xavier_uniform_(module.weight)
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, (nn.BatchNorm2d, nn.LayerNorm)):
        nn.init.ones_(module.weight)
        nn.init.zeros_(module.bias)

model = SimpleCNN()
model.apply(init_weights)
print("权重初始化完成")

# 验证初始化分布
w = model.features[0].weight
print(f"Conv2d weight: mean={w.mean().item():.4f}, std={w.std().item():.4f}")

### 12.5 梯度累积（大 batch 模拟）

**核心思想：** 当 GPU 显存不足以容纳大 batch 时，将一个大 batch 分成多个小 batch，累积梯度后再更新参数。

**操作流程：**
1. `optimizer.zero_grad()` → 在累积开始时清零梯度
2. 对每个小 batch：`loss / accum_steps → backward()`（loss 除以累积步数，使梯度平均值与大 batch 等价）
3. 每累积 `accum_steps` 步 → `optimizer.step()` → `zero_grad()`
4. 处理最后一个不完整的累积周期

**面试要点：**
1. **数学等价性**：梯度累积 `accum_steps=4` + `batch_size=8` 等价于 `batch_size=32`（梯度相同）
2. **显存节省**：每次只加载一个小 batch 的数据到显存
3. **使用场景**：大模型训练（如 ViT、GPT）、高分辨率输入（如 1024×1024 图像）

**注意事项：**
- **必须**将 loss 除以 `accum_steps`，否则梯度是被放大了 accum_steps 倍
- 梯度累积不等于 data parallelism（不加速训练，只是省显存）
- BN 统计量仍然基于小 batch 计算（与真正的大 batch 不完全等价），可考虑 SyncBN
- 最后一个 batch 如果不满 `accum_steps`，仍然要 step（不能丢梯度）

In [ ]:
def train_with_gradient_accumulation(model, loader, criterion, optimizer, device, accum_steps=4):
    """梯度累积 — 用小 batch 模拟大 batch"""
    model.train()
    optimizer.zero_grad()
    
    for i, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels) / accum_steps  # 缩放损失
        loss.backward()
        
        if (i + 1) % accum_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

    # 处理剩余梯度
    if (i + 1) % accum_steps != 0:
        optimizer.step()
        optimizer.zero_grad()

print("梯度累积模板已定义")

In [ ]:
print("=" * 50)
print("题库准备完毕！共 12 大类 30+ 手撕题")
print("建议：每道题先自己写，再对照答案")
print("=" * 50)